# 02 — Candidate-table contract and validation

**Owner:** Person B — Modelling Lead  
**Purpose:** Validate Person A's candidate-generation output before any model is trained.

This notebook intentionally does not manufacture candidates or labels. It checks the agreed interface and highlights data issues that must be resolved with Person A.

## Required input contract

One row per candidate road arc for one GPS observation:

| Column | Meaning |
|---|---|
| `trip_id` | Stable trip identifier; used for group-aware splitting |
| `observation_idx` | GPS observation index, with one documented indexing convention |
| `candidate_arc_id` | Arc ID from that trip's `.arcs` file |
| `is_match` | Binary label: 1 for ground-truth matched candidate, 0 otherwise |
| other numeric columns | Candidate features agreed and documented by Person A |

**Critical:** the dataset guide says not to assume a GPS row corresponds directly to a road-network row. Person A must document how `.route` labels are aligned to GPS observations. Do not proceed if this is unclear.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
TRAIN_CANDIDATES = REPO_ROOT / "data/processed/train_candidates.csv"
STUDENT_CANDIDATES = REPO_ROOT / "data/processed/student_test_candidates.csv"
REQUIRED = {"trip_id", "observation_idx", "candidate_arc_id", "is_match"}
IDENTIFIERS = {"trip_id", "observation_idx", "candidate_arc_id"}
TARGET = "is_match"

print("Repository root:", REPO_ROOT)
print("Training candidate file:", TRAIN_CANDIDATES)
print("Student-test candidate file:", STUDENT_CANDIDATES)

In [ ]:
def load_and_validate(path: Path, require_labels=True):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Ask Person A to provide the candidate table using the agreed schema."
        )
    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(f"{path.name} is empty")
    missing = REQUIRED - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    if df[list(IDENTIFIERS)].isna().any().any():
        raise ValueError("Identifier columns contain missing values")
    if require_labels:
        if df[TARGET].isna().any():
            raise ValueError("Target contains missing values")
        labels = set(df[TARGET].unique())
        if not labels.issubset({0, 1, False, True}):
            raise ValueError(f"Target must be binary 0/1; found {labels}")
        df[TARGET] = df[TARGET].astype(int)
    return df

train_df = load_and_validate(TRAIN_CANDIDATES)
student_df = load_and_validate(STUDENT_CANDIDATES)
print("Training rows:", len(train_df), "| trips:", train_df.trip_id.nunique())
print("Student-test rows:", len(student_df), "| trips:", student_df.trip_id.nunique())
display(train_df.head())

In [ ]:
# Identify numeric model features. IDs and labels must never be predictors.
feature_cols = [
    c for c in train_df.select_dtypes(include=np.number).columns
    if c not in IDENTIFIERS | {TARGET}
]
if not feature_cols:
    raise ValueError("No numeric feature columns found. Agree and add numeric candidate features with Person A.")
missing_student_features = sorted(set(feature_cols) - set(student_df.columns))
if missing_student_features:
    raise ValueError(f"Student-test table lacks training features: {missing_student_features}")
non_numeric_features = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(student_df[c])]
if non_numeric_features:
    raise ValueError(f"Non-numeric feature columns need encoding: {non_numeric_features}")

print("Numeric features:", feature_cols)
print("Training label distribution:")
display(train_df[TARGET].value_counts(dropna=False).rename_axis("is_match").to_frame("rows"))
print("Positive-label rate by trip (first 10):")
display(train_df.groupby("trip_id")[TARGET].mean().head(10))

In [ ]:
# One candidate should not be duplicated for the same trip/observation/arc.
dup_mask = train_df.duplicated(["trip_id", "observation_idx", "candidate_arc_id"], keep=False)
print("Duplicate candidate keys:", int(dup_mask.sum()))

# Every GPS observation represented in the candidate table should have at least one candidate.
obs_counts = train_df.groupby(["trip_id", "observation_idx"]).size()
print("Training candidate rows per observation:")
display(obs_counts.describe(percentiles=[.5, .9, .95, .99]).to_frame("candidate_count"))

# Candidate coverage: fraction of observations with at least one positive candidate.
coverage = train_df.groupby(["trip_id", "observation_idx"])[TARGET].max()
print("Observations with at least one positive candidate:", f"{coverage.mean():.3%}")
print("Observations with no positive candidate:", int((coverage == 0).sum()))

# Save an explicit feature manifest for downstream notebooks.
manifest = {
    "feature_columns": feature_cols,
    "target_column": TARGET,
    "group_column": "trip_id",
    "candidate_key": ["trip_id", "observation_idx", "candidate_arc_id"],
    "label_alignment_owner": "Person A — must be documented and verified",
}
Path("configs").mkdir(exist_ok=True)
Path("configs/person_b_feature_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved configs/person_b_feature_manifest.json")

## Stop/go checklist

Proceed to model training only when all are true:

- [ ] Person A confirms the label-alignment method from reference `.route` to GPS observations.
- [ ] Both candidate CSVs use the same feature definitions and units.
- [ ] `trip_id` is stable and correctly identifies each trip.
- [ ] `observation_idx` indexing convention is documented (zero-based or one-based).
- [ ] Candidate arc IDs refer to the corresponding trip's `.arcs` file.
- [ ] Positive-label coverage is plausible; zero-positive observations are investigated.
- [ ] Duplicate candidate keys are explained or removed upstream.

Do not silently drop suspicious rows or change labels in this notebook. Resolve issues with Person A and document the decision.